# Aswenna Agricultural Marketplace - NLP Chatbot Modeling & QA Pipeline

This notebook covers the complete end-to-end pipeline for the **Aswenna NLP Chatbot** agricultural knowledge retrieval system:
1. **Parquet and Text Dataset Extraction & Standardization**
2. **Text Cleaning & Normalization**
3. **Semantic Keyword TF-IDF & Cosine Similarity Knowledge Retrieval Model**
4. **Intent Classification** (Predicting request intent groups like Pests, Soil, Fertilizer, general)
5. **Serialization** (Saving models and knowledge databases for production API inference)

In [1]:
import os
import pandas as pd
import numpy as np
import re
import joblib
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

# Create directories for processed datasets and serialized models
os.makedirs('datasets/processed', exist_ok=True)
os.makedirs('models', exist_ok=True)

print("NLP libraries loaded and directories initialized.")

NLP libraries loaded and directories initialized.


## 1. Load and Parse Raw Datasets

In [2]:
# -----------------------------------------------------
# 1.1 ShuklaShreyansh Parquet Dataset
# -----------------------------------------------------
shukla_df = pd.read_parquet('datasets/raw/ShuklaShreyanshAgriculture-QA/train-00000-of-00001.parquet')

shukla_qa = []
for conv in shukla_df['conversations']:
    q, a = "", ""
    for turn in conv:
        if turn['role'] == 'user':
            q = turn['content']
        elif turn['role'] == 'assistant':
            a = turn['content']
            if q and a:
                shukla_qa.append({'question': q, 'answer': a, 'source': 'shukla'})
shukla_processed = pd.DataFrame(shukla_qa)
print(f"Extracted {len(shukla_processed)} QA pairs from Shukla dataset.")

# -----------------------------------------------------
# 1.2 Rokayon AgriQA Parquet Dataset
# -----------------------------------------------------
rokayon_df = pd.read_parquet('datasets/raw/rokayonAgriQA_Multilingual_Dataset/train-00000-of-00001.parquet')
# Filter by English
rokayon_en = rokayon_df[rokayon_df['lang'] == 'en'][['question', 'answer']].copy()
rokayon_en['source'] = 'rokayon'
print(f"Extracted {len(rokayon_en)} English QA pairs from Rokayon dataset.")

# -----------------------------------------------------
# 1.3 Yuvraj Soil Text Dataset
# -----------------------------------------------------
yuvraj_df = pd.read_csv('datasets/raw/YuvrajSingh9886Agriculture-Soil-QA-Pairs-Dataset/qna-dataset-farmgenie-soil-v2.txt')
yuvraj_processed = pd.DataFrame({
    'question': yuvraj_df['QUESTION.question'],
    'answer': yuvraj_df['ANSWER'],
    'source': 'yuvraj'
}).dropna()
print(f"Extracted {len(yuvraj_processed)} QA pairs from Yuvraj dataset.")

# -----------------------------------------------------
# 1.4 Talha Text Dataset
# -----------------------------------------------------
talha_df = pd.read_csv('datasets/raw/talhakkagriculture-qa/merged_agriculture_dataset.txt')
talha_processed = pd.DataFrame({
    'question': talha_df['question'],
    'answer': talha_df['answer'],
    'source': 'talha'
}).dropna()
print(f"Extracted {len(talha_processed)} QA pairs from Talha dataset.")

Extracted 28531 QA pairs from Shukla dataset.
Extracted 18092 English QA pairs from Rokayon dataset.
Extracted 3447 QA pairs from Yuvraj dataset.
Extracted 25410 QA pairs from Talha dataset.


## 2. Combine and Clean Dataset

In [3]:
combined_df = pd.concat([shukla_processed, rokayon_en, yuvraj_processed, talha_processed], ignore_index=True)
combined_df = combined_df.drop_duplicates(subset=['question'])
combined_df = combined_df.dropna()

def clean_text(text):
    text = re.sub(r'<[^>]*>', '', str(text))  # Strip HTML
    text = re.sub(r'\s+', ' ', text).strip()  # Normalize whitespace
    return text

combined_df['question'] = combined_df['question'].apply(clean_text)
combined_df['answer'] = combined_df['answer'].apply(clean_text)

# Assign intents based on simple rules/keywords for demonstration
def extract_intent(row):
    q = row['question'].lower()
    if 'pest' in q or 'insect' in q or 'bug' in q or 'spray' in q:
        return 'pest_control'
    elif 'soil' in q or 'clay' in q or 'sand' in q or 'moisture' in q:
        return 'soil_management'
    elif 'fertilizer' in q or 'urea' in q or 'dap' in q or 'npk' in q:
        return 'fertilizer_scheduling'
    elif 'irrigate' in q or 'water' in q or 'rain' in q:
        return 'irrigation'
    else:
        return 'general_crop_advice'

combined_df['intent'] = combined_df.apply(extract_intent, axis=1)
combined_df.to_csv('datasets/processed/cleaned_agriculture_qa.csv', index=False)
print(f"Unified dataset contains {len(combined_df)} QA pairs.")
print(combined_df['intent'].value_counts())

Unified dataset contains 28013 QA pairs.
intent
general_crop_advice      22238
fertilizer_scheduling     1843
soil_management           1838
pest_control              1360
irrigation                 734
Name: count, dtype: int64


## 3. Train Intent Classification Model

In [4]:
X = combined_df['question']
y = combined_df['intent']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

vectorizer = TfidfVectorizer(max_features=2500, stop_words='english')
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

print("Training intent classifier...")
clf_intent = LogisticRegression(max_iter=500, random_state=42)
clf_intent.fit(X_train_vec, y_train)

y_pred = clf_intent.predict(X_test_vec)
print(f"Intent Classifier Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(classification_report(y_test, y_pred))

Training intent classifier...
Intent Classifier Accuracy: 0.9825
                       precision    recall  f1-score   support

fertilizer_scheduling       0.99      0.97      0.98       369
  general_crop_advice       0.98      1.00      0.99      4448
           irrigation       0.99      0.76      0.86       147
         pest_control       1.00      0.88      0.94       272
      soil_management       0.99      0.95      0.97       367

             accuracy                           0.98      5603
            macro avg       0.99      0.91      0.95      5603
         weighted avg       0.98      0.98      0.98      5603



## 4. Semantic Search / Knowledge Base Indexing (RAG Simulation)

Building a semantic vector database using TF-IDF matrices to retrieve answers based on cosine similarity.

In [5]:
# Vectorize all questions in our database
kb_vectorizer = TfidfVectorizer(stop_words='english')
kb_matrix = kb_vectorizer.fit_transform(combined_df['question'])

def retrieve_answer(query, threshold=0.2):
    query_vec = kb_vectorizer.transform([query])
    similarities = cosine_similarity(query_vec, kb_matrix).flatten()
    best_idx = np.argmax(similarities)
    if similarities[best_idx] >= threshold:
        return combined_df.iloc[best_idx]['answer'], similarities[best_idx]
    return "I'm having trouble finding a precise match in my database. Please rephrase or contact an agricultural advisor.", similarities[best_idx]

# Test query
test_query = "How does crop rotation help prevent soil erosion?"
answer, score = retrieve_answer(test_query)
print(f"Query: {test_query}")
print(f"Best Match Similarity Score: {score:.4f}")
print(f"Retrieved Answer: {answer}")

Query: How does crop rotation help prevent soil erosion?
Best Match Similarity Score: 0.5744
Retrieved Answer: planting cover crops, using conservation tillage practices, and building contour terraces. It's also important to avoid overgrazing and to maintain vegetation along streams and other waterways to prevent bank erosion.


## 5. Serialize Models and Database

Saving the trained models and database lists for inference serving.

In [6]:
joblib.dump(vectorizer, 'models/intent_vectorizer.pkl')
joblib.dump(clf_intent, 'models/intent_classifier.pkl')

joblib.dump(kb_vectorizer, 'models/kb_vectorizer.pkl')
joblib.dump(kb_matrix, 'models/kb_matrix.pkl')
combined_df[['question', 'answer', 'intent']].to_parquet('models/kb_database.parquet', index=False)

print("All NLP models and datasets saved successfully.")

All NLP models and datasets saved successfully.
